In [2]:

import numpy as np
from pathlib import Path

# Locate project directory
project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Load the prepared dataset
dataset_path = project_dir / "results" / "raw_window_flattened_split.npz"

data = np.load(dataset_path)

X_train_rf = data["X_train"]
X_test_rf = data["X_test"]
y_train_rf = data["y_train"]
y_test_rf = data["y_test"]

print("Training shape:", X_train_rf.shape)
print("Testing shape:", X_test_rf.shape)
print("Training labels:", y_train_rf.shape)
print("Testing labels:", y_test_rf.shape)


Training shape: (500, 1800)
Testing shape: (127, 1800)
Training labels: (500,)
Testing labels: (127,)



### Train Random Forest Regressor


In [3]:

from sklearn.ensemble import RandomForestRegressor

# Initialize Random Forest
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(X_train_rf, y_train_rf)

print("Random Forest training completed!")
print("Number of trees:", rf_model.n_estimators)
print("Training samples:", X_train_rf.shape[0])
print("Input features:", X_train_rf.shape[1])


Random Forest training completed!
Number of trees: 200
Training samples: 500
Input features: 1800



### Evaluate Random Forest Regression


In [4]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predict fatigue scores for 127 test windows
y_pred_rf = rf_model.predict(X_test_rf)

# Keep predictions within the 0–10 fatigue scale
y_pred_rf = np.clip(y_pred_rf, 0, 10)

# Calculate evaluation metrics
rf_mae = mean_absolute_error(y_test_rf, y_pred_rf)
rf_mse = mean_squared_error(y_test_rf, y_pred_rf)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test_rf, y_pred_rf)

print(f"MAE: {rf_mae:.4f}")
print(f"MSE: {rf_mse:.4f}")
print(f"RMSE: {rf_rmse:.4f}")
print(f"R² Score: {rf_r2:.4f}")


MAE: 2.0015
MSE: 5.0559
RMSE: 2.2485
R² Score: 0.1456



### Random Forest Classification Accuracy


In [5]:

from sklearn.metrics import accuracy_score

# Convert actual and predicted fatigue scores into classes
y_test_rf_levels = np.digitize(y_test_rf, bins=[4, 7])
y_pred_rf_levels = np.digitize(y_pred_rf, bins=[4, 7])

# Calculate classification accuracy
rf_accuracy = accuracy_score(
    y_test_rf_levels,
    y_pred_rf_levels
)

print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")


Random Forest Accuracy: 55.12%



### Evaluate Random Forest High Fatigue Recall


In [6]:

# Identify actual High Fatigue windows
high_mask_rf = y_test_rf_levels == 2

# Count actual and correctly detected cases
total_high_rf = high_mask_rf.sum()
detected_high_rf = (
    y_pred_rf_levels[high_mask_rf] == 2
).sum()

# Calculate High Fatigue Recall
rf_high_recall = (
    detected_high_rf / total_high_rf
    if total_high_rf > 0 else 0
)

print("Actual High Fatigue Windows:", total_high_rf)
print("Correctly Detected:", detected_high_rf)
print(f"High Fatigue Recall: {rf_high_recall * 100:.2f}%")


Actual High Fatigue Windows: 19
Correctly Detected: 0
High Fatigue Recall: 0.00%



## Random Forest — 3-Minute Window Flattened Model

### Model Configuration

- **Window Duration:** 3 minutes (180 seconds)
- **Readings per Window:** 360
- **Sensor Signals:** 5
- **Input Features:** 1,800 (Flattened Raw Readings)
- **Statistical Feature Extraction:** None
- **Training Windows:** 500
- **Testing Windows:** 127
- **Number of Trees:** 200
- **Maximum Tree Depth:** 8
- **Minimum Samples per Leaf:** 5

### Model Evaluation Results

- **Classification Accuracy:** 55.12%
- **MAE (Mean Absolute Error):** 2.0015
- **MSE (Mean Squared Error):** 5.0559
- **RMSE (Root Mean Squared Error):** 2.2485
- **R² Score:** 0.1456
- **Actual High Fatigue Windows:** 19
- **Correctly Detected High Fatigue Windows:** 0
- **High Fatigue Recall:** 0.00%



## Save Random Forest 3-Minute Window Flattened Model


In [7]:

import joblib

model_dir = project_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "random_forest_3min_flattened.joblib"

joblib.dump(rf_model, model_path)

print("Random Forest model saved successfully!")
print("Location:", model_path)


Random Forest model saved successfully!
Location: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\models\random_forest_3min_flattened.joblib
